In [2]:
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import pycuda.autoinit
from utils.yolo import TRT_YOLO
import cv2
import PIL.Image
import numpy as np
import time

# ==========================================
# 1. 道路跟隨模型前處理
# ==========================================
device = torch.device('cuda')
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road(image):
    """道路模型使用 224x224；即使 Camera 開 416x416，這裡也會縮回 224。"""
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

print("開始載入模型，這可能需要幾分鐘...")

# ==========================================
# 2. 載入道路跟隨 TensorRT 模型
# ==========================================
model_road = TRTModule()
model_road.load_state_dict(torch.load('../Final_team1/road_following_model_/best_steering_model_xy_trt.pth'))
print("✅ 道路跟隨模型載入完成")

# ==========================================
# 3. 載入 YOLO TensorRT 號誌模型
# ==========================================
# Project06 原本使用 yolov4-tiny-416；若你們的檔名是 yolov4-tiny-new.trt，會自動改用它。
yolo_model_name = '../Final_team1/yolo/yolov4-tiny-416'
try:
    trt_yolo = TRT_YOLO(yolo_model_name, (416, 416), 4)
    print(f"✅ 路牌辨識模型載入完成：{yolo_model_name}.trt")
except Exception as e:
    print(f"⚠️ 無法載入 {yolo_model_name}.trt，嘗試 yolov4-tiny-new.trt")
    yolo_model_name = 'yolov4-tiny-new'
    trt_yolo = TRT_YOLO(yolo_model_name, (416, 416), 4)
    print(f"✅ 路牌辨識模型載入完成：{yolo_model_name}.trt")


開始載入模型，這可能需要幾分鐘...
✅ 道路跟隨模型載入完成
⚠️ 無法載入 ../Final_team1/yolo/yolov4-tiny-416.trt，嘗試 yolov4-tiny-new.trt
✅ 路牌辨識模型載入完成：yolov4-tiny-new.trt


Exception ignored in: <bound method TRT_YOLO.__del__ of <utils.yolo.TRT_YOLO object at 0x7f9ab0b0f0>>
Traceback (most recent call last):
  File "/workspace/Nvidia/jetbot/notebooks/road_following/trt_yolv4-tiny-master/utils/yolo.py", line 266, in __del__
    del self.outputs
AttributeError: outputs


In [8]:
from jetbot import Camera, bgr8_to_jpeg
from jetbot import Robot
import ipywidgets
import traitlets
from IPython.display import display

# ==========================================
# 初始化設備
# ==========================================
robot = Robot()

# ⚠️ 重要修正：YOLO 是 416x416 訓練 / 轉 TRT，因此 Camera 建議使用 416x416。
# 道路模型會在 preprocess_road() 裡自行 resize 成 224x224。
camera = Camera.instance(width=416, height=416, fps=10)
try:
    camera.running = True
except Exception:
    pass

# ==========================================
# 道路跟隨參數
# ==========================================
speed_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.20, description='speed gain')
steering_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.10, description='steering gain')
steering_dgain_slider = ipywidgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.01, description='steering kd')
steering_bias_slider = ipywidgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=-0.03, description='steering bias')

# ==========================================
# 號誌辨識參數
# ==========================================
yolo_conf_slider = ipywidgets.FloatSlider(min=0.05, max=0.9, step=0.05, value=0.50, description='YOLO conf')
yolo_interval_slider = ipywidgets.IntSlider(min=1, max=30, step=1, value=3, description='YOLO間隔')
alert_width_slider = ipywidgets.IntSlider(min=5, max=200, step=5, value=30, description='觸發寬度')

# Slow 標誌慢行參數
# 注意：JetBot 馬達有啟動門檻，太低會看似停止；slow_ratio 不建議低於 0.75。
slow_ratio_slider = ipywidgets.FloatSlider(min=0.50, max=1.00, step=0.05, value=0.85, description='Slow倍率')
slow_min_slider = ipywidgets.FloatSlider(min=0.00, max=0.30, step=0.01, value=0.10, description='Slow最低速')

# ==========================================
# 狀態顯示
# ==========================================
status_label = ipywidgets.HTML(value='<b>status:</b> waiting')
road_label = ipywidgets.HTML(value='road: -')
sign_label = ipywidgets.HTML(value='sign: -')

# raw_image_widget：第二格顯示「相機原始即時影像」
# image_widget：第五格執行後顯示「道路點 + 號誌框 + 狀態文字」
raw_image_widget = ipywidgets.Image(format='jpeg', width=320, height=320)
image_widget = ipywidgets.Image(format='jpeg', width=416, height=416)

try:
    camera_link.unlink()
except Exception:
    pass

camera_link = traitlets.dlink((camera, 'value'), (raw_image_widget, 'value'), transform=bgr8_to_jpeg)

print("請微調以下參數以適應您的場地：")
display(ipywidgets.VBox([
    ipywidgets.HBox([speed_gain_slider, steering_gain_slider]),
    ipywidgets.HBox([steering_dgain_slider, steering_bias_slider]),
    ipywidgets.HBox([yolo_conf_slider, yolo_interval_slider, alert_width_slider]),
    ipywidgets.HBox([slow_ratio_slider, slow_min_slider]),
    status_label,
    road_label,
    sign_label,
    ipywidgets.HBox([
        ipywidgets.VBox([ipywidgets.HTML('<b>Camera 即時原始畫面</b>'), raw_image_widget]),
        ipywidgets.VBox([ipywidgets.HTML('<b>辨識結果畫面</b>'), image_widget])
    ])
]))


請微調以下參數以適應您的場地：


In [9]:
# Class ID 對照：Road close(0), Slow(1), Railway(2), Stop(3)
CLASS_NAMES = {
    0: 'road close',
    1: 'slow',
    2: 'railway',
    3: 'stop'
}

def getNearest(signs, current_time, stop_ignore_time, railway_ignore_time):
    """從已依寬度排序的 signs 中，找出目前可以觸發的最近路牌。"""
    if not len(signs):
        return [0, -1, 0.0]
        
    cls_id = int(signs[0][1])
    
    if (cls_id == 3 and (current_time > stop_ignore_time)) \
        or (cls_id == 2 and (current_time > railway_ignore_time)) \
        or (cls_id in [0, 1]): 
        return signs[0]

    return getNearest(signs[1:], current_time, stop_ignore_time, railway_ignore_time) if len(signs) >= 2 else [0, -1, 0.0]


In [10]:
import time
import numpy as np
import cv2

# ==========================================
# 1. 清除背景 observe，避免和第五格手動 while 迴圈打架
# ==========================================
try:
    camera.unobserve_all()
except Exception:
    pass

# 第二格的 raw camera link 如果被 unobserve_all 影響，重新接上
try:
    camera_link.unlink()
except Exception:
    pass
try:
    camera_link = traitlets.dlink((camera, 'value'), (raw_image_widget, 'value'), transform=bgr8_to_jpeg)
except Exception:
    pass

# ==========================================
# 2. 全域變數初始化
# ==========================================
stop_ignore = 0
railway_ignore = 0
stop_until_time = 0
slow_until_time = 0
angle_last = 0.0
frame_count = 0
is_processing = False
last_sign_text = 'none'
last_sign_time = 0

# 若車子左右方向相反，可在這裡改成 True
REVERSE_STEERING = False
SWAP_MOTORS = False


def clamp_motor(v):
    return max(min(float(v), 1.0), 0.0)


def draw_text(img, text, y, color=(0, 255, 255)):
    cv2.putText(img, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


def execute(change):
    global angle_last, is_processing, frame_count
    global stop_until_time, stop_ignore, railway_ignore, slow_until_time
    global last_sign_text, last_sign_time
    
    if is_processing:
        return
    is_processing = True

    try:
        t = time.time()
        image = change['new']
        if image is None:
            status_label.value = '<b>status:</b> camera.value is None'
            return

        img_draw = image.copy()
        h, w = img_draw.shape[:2]
        status_text = 'road following'
        sign_text = 'none'
        
        # ====================================================
        # 狀態機 1：強制停止期間
        # ====================================================
        if t < stop_until_time:
            robot.stop()
            remain = stop_until_time - t
            draw_text(img_draw, f"STOPPING... {remain:.1f}s", 40, (0, 0, 255))
            draw_text(img_draw, f"last sign: {last_sign_text}", 70, (0, 0, 255))
            image_widget.value = bgr8_to_jpeg(img_draw)
            status_label.value = f'<b>status:</b> STOPPING {remain:.1f}s'
            sign_label.value = f'sign: {last_sign_text}'
            return

        # ====================================================
        # 任務一：道路跟隨
        # ====================================================
        xy = model_road(preprocess_road(image)).detach().float().cpu().numpy().flatten()
        x = float(xy[0])
        y = float((0.5 - xy[1]) / 2.0)
        
        # 將道路模型輸出點畫在 416x416 畫面上
        point_x = int((xy[0] + 1.0) / 2.0 * w)
        point_y = int((xy[1] + 1.0) / 2.0 * h)
        point_x = max(0, min(w - 1, point_x))
        point_y = max(0, min(h - 1, point_y))
        cv2.circle(img_draw, (point_x, point_y), 10, (0, 255, 0), -1)
        
        angle = np.arctan2(x, y)
        pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
        angle_last = angle
        
        steering = float(pid + steering_bias_slider.value)
        if REVERSE_STEERING:
            steering = -steering
            
        base_speed = float(speed_gain_slider.value)
        speed_left = base_speed + steering
        speed_right = base_speed - steering

        # ====================================================
        # 任務二：號誌辨識 YOLO
        # ====================================================
        yolo_interval = max(1, int(yolo_interval_slider.value))
        run_yolo = (frame_count % yolo_interval == 0)
        frame_count += 1
        
        if run_yolo:
            boxes, confs, clss = trt_yolo.detect(image, conf_th=float(yolo_conf_slider.value))
            signs = []
            
            for box, conf, cls in zip(boxes, confs, clss):
                x1, y1, x2, y2 = [int(v) for v in box]
                cls_id = int(cls)
                conf_v = float(conf)
                width = int(x2 - x1)
                signs.append([width, cls_id, conf_v])
                name = CLASS_NAMES.get(cls_id, str(cls_id))
                cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img_draw, f"{name} {conf_v:.2f} W:{width}",
                            (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, (0, 255, 255), 2)

            signs.sort(reverse=True, key=lambda s: s[0])
            sign = getNearest(signs, t, stop_ignore, railway_ignore)
            alert_width = int(alert_width_slider.value)
            
            if sign[1] != -1:
                sign_name = CLASS_NAMES.get(int(sign[1]), str(sign[1]))
                sign_text = f"{sign_name}, width={sign[0]}, conf={sign[2]:.2f}"
                last_sign_text = sign_text
                last_sign_time = t
            
            # --- 判斷號誌邏輯 ---
            if int(sign[1]) == 3 and sign[0] > alert_width:
                # Stop：立刻停，不等下一圈
                robot.stop()
                stop_until_time = t + 3.0
                stop_ignore = t + 10.0
                status_text = 'STOP sign -> stop 3s'
                draw_text(img_draw, status_text, 40, (0, 0, 255))
                image_widget.value = bgr8_to_jpeg(img_draw)
                status_label.value = f'<b>status:</b> {status_text}'
                sign_label.value = f'sign: {sign_text}'
                print(f"🛑 Stop: {sign_text}")
                return
                
            elif int(sign[1]) == 2 and sign[0] > max(5, alert_width - 20):
                # Railway：立刻停，不等下一圈
                robot.stop()
                stop_until_time = t + 5.0
                railway_ignore = t + 10.0
                status_text = 'RAILWAY sign -> stop 5s'
                draw_text(img_draw, status_text, 40, (0, 0, 255))
                image_widget.value = bgr8_to_jpeg(img_draw)
                status_label.value = f'<b>status:</b> {status_text}'
                sign_label.value = f'sign: {sign_text}'
                print(f"🚆 Railway: {sign_text}")
                return
                
            elif int(sign[1]) == 1 and sign[0] > alert_width:
                # Slow：進入慢行模式。實際慢行速度由第二格的 Slow倍率 / Slow最低速 控制。
                slow_until_time = t + 2.0
                status_text = 'SLOW sign -> slow 2s'
                print(f"⚠️ Slow: {sign_text}")
                
            elif int(sign[1]) == 0 and sign[0] > alert_width:
                robot.stop()
                stop_until_time = t + 999.0
                status_text = 'ROAD CLOSE sign -> stop'
                draw_text(img_draw, status_text, 40, (0, 0, 255))
                image_widget.value = bgr8_to_jpeg(img_draw)
                status_label.value = f'<b>status:</b> {status_text}'
                sign_label.value = f'sign: {sign_text}'
                print(f"⛔ Road close: {sign_text}")
                return
        else:
            # 沒跑 YOLO 的幀，保留最近一次標誌狀態幾秒，方便觀察
            if t - last_sign_time < 2.0:
                sign_text = last_sign_text

        # ====================================================
        # 任務三：馬達輸出
        # ====================================================
        # Slow 標誌不要直接把左右輪一起乘上很小倍率，
        # 否則 JetBot 馬達可能低於啟動門檻而停住。
        # 改成：慢行時提高最低前進速度，但保留道路跟隨的轉向修正。
        if t < slow_until_time:
            slow_ratio = float(slow_ratio_slider.value)
            slow_min = float(slow_min_slider.value)
            slow_base_speed = max(base_speed * slow_ratio, slow_min)
            final_left = clamp_motor(slow_base_speed + steering)
            final_right = clamp_motor(slow_base_speed - steering)
            status_text = f'slow mode ratio={slow_ratio:.2f}, min={slow_min:.2f}'
        else:
            final_left = clamp_motor(speed_left)
            final_right = clamp_motor(speed_right)

        if SWAP_MOTORS:
            final_left, final_right = final_right, final_left

        robot.left_motor.value = final_left
        robot.right_motor.value = final_right
        
        draw_text(img_draw, f"mode: {status_text}", 30, (255, 255, 0))
        draw_text(img_draw, f"road x={x:.2f}, y={y:.2f}, steer={steering:.2f}", 60, (0, 255, 0))
        draw_text(img_draw, f"motor L={final_left:.2f}, R={final_right:.2f}", 90, (0, 255, 0))
        draw_text(img_draw, f"sign: {sign_text}", 120, (0, 255, 255))
        image_widget.value = bgr8_to_jpeg(img_draw)
        
        status_label.value = f'<b>status:</b> {status_text}'
        road_label.value = f'road: x={x:.3f}, y={y:.3f}, steering={steering:.3f}, L={final_left:.3f}, R={final_right:.3f}'
        sign_label.value = f'sign: {sign_text}'

    except Exception as e:
        robot.stop()
        status_label.value = f'<b>status:</b> ERROR {repr(e)}'
        print('execute error:', repr(e))
        raise
    finally:
        is_processing = False


In [ ]:
import time
from IPython.display import clear_output

# ==========================================
# 第五格：保留原本的 while True 手動迴圈模式
# 按上方停止/Interrupt Kernel 後會進入 KeyboardInterrupt 並停車
# ==========================================
try:
    camera.running = True
except Exception:
    pass

# 起步推力，克服靜摩擦；太衝可改成 0.20
robot.set_motors(0.20, 0.20)
time.sleep(0.15)
robot.stop()
time.sleep(0.05)

print("🚗 引擎啟動，進入無窮迴圈模式！")
print("提示：辨識結果會顯示在第二格的『辨識結果畫面』，停止請按 Jupyter 的 Interrupt/Stop。Slow 太慢請調高第二格的 Slow倍率或 Slow最低速。")

try:
    while True:
        execute({'new': camera.value})
        # 這個 sleep 很重要：讓 Jupyter 有時間更新 widget，也避免 CPU/GPU 被 while 迴圈吃滿
        time.sleep(0.03)
except KeyboardInterrupt:
    robot.stop()
    status_label.value = '<b>status:</b> manual stop'
    print("🛑 已手動停止！")


🚗 引擎啟動，進入無窮迴圈模式！
提示：辨識結果會顯示在第二格的『辨識結果畫面』，停止請按 Jupyter 的 Interrupt/Stop。Slow 太慢請調高第二格的 Slow倍率或 Slow最低速。
🚆 Railway: railway, width=60, conf=0.69
🚆 Railway: railway, width=18, conf=0.85


In [6]:
# 第六格：安全停車
speed_gain_slider.value = 0.0
robot.stop()
status_label.value = '<b>status:</b> stopped'
print("🛑 車輛已安全停止")

# 測試完全結束後才執行 camera.stop()
# camera.stop()


🛑 車輛已安全停止


In [7]:
# 第七格：完全關閉 Camera；要重新測試時需從第二格重新初始化 Camera
try:
    camera_link.unlink()
except Exception:
    pass
robot.stop()
camera.stop()
print("camera stopped")


camera stopped


In [15]:
!free -m

              total        used        free      shared  buff/cache   available
Mem:           3964        3307         217           4         439         514
Swap:          6078        1048        5029
